In [22]:
1 + 1


[1] 2

In [23]:
# Loading of da library

library(dplyr)
library(ggplot2)
library(caret)
library(tidyverse)
library(arrow)
library(skimr)
library(janitor)

# Set the seed for reproducibility
set.seed(123)


In [24]:
# Data Baseline info Check
baselineInfo <- function(df, file) {
  print(paste("File:", file))
  print(paste("Number of rows:", nrow(df)))
  print(paste("Number of columns:", ncol(df)))
  print(paste("Column names:", paste(colnames(df), collapse = ", ")))
  print(paste("Data types:", paste(sapply(df, class), collapse = ", ")))
  print(paste("Missing values:", sum(is.na(df))))
  print(paste("Unique values per column:"))
  print(sapply(df, function(x) length(unique(x))))
  head(df, n = 3)
}


In [25]:
# check Fraud Data
checkFraudData <- function(df) {
  # Make the fraud_bool column as integer
  df <- df %>%
    mutate(fraud_bool = as.integer(.data[["fraud_bool"]]))

  # check for values that are not 0 or 1 in the fraud_bool column
  invalid_values <- df %>%
    filter(fraud_bool != 0 & fraud_bool != 1 | is.na(fraud_bool))

  if (nrow(invalid_values) > 0) {
    print("Invalid values found in fraud_bool column:")
    print(invalid_values)
  } else {
    print("All values in fraud_bool column are valid (0 or 1).")
  }
}


In [26]:
# Based on Data Sheet https://github.com/feedzai/bank-account-fraud/blob/main/documents/datasheet.pdf

set_missing_to_na <- function(df) {
  minus1_is_missing <- c(
    "prev_address_months_count",
    "current_address_months_count",
    "bank_months_count",
    "session_length_in_minutes",
    "device_distinct_emails"
  )

  binary_cols <- c(
    "email_is_free", "phone_home_valid", "phone_mobile_valid",
    "has_other_cards", "foreign_request", "keep_alive_session",
    "device_fraud_count", "fraud_bool"
  )

  df %>%
    dplyr::mutate(
      dplyr::across(dplyr::any_of(minus1_is_missing), ~ dplyr::na_if(., -1)),
      intended_balcon_amount = dplyr::if_else(
        intended_balcon_amount < 0, NA_real_, as.numeric(intended_balcon_amount)
      ),
      dplyr::across(
        dplyr::any_of(binary_cols),
        ~ dplyr::if_else(. %in% c(0, 1), as.integer(.), NA_integer_)
      )
    )
}

# Data Quality Check
dataQualityCheck <- function(df) {
  # Update missing values based on the datasheet
  df <- set_missing_to_na(df)


  # Check for missing values
  missing_values <- sapply(df, function(x) sum(is.na(x)))
  print("Missing values per column:")
  print(missing_values)

  # Check for duplicate rows
  duplicate_rows <- df[duplicated(df), ]
  if (nrow(duplicate_rows) > 0) {
    print(paste("Number of duplicate rows:", nrow(duplicate_rows)))
  } else {
    print("No duplicate rows found.")
  }

  # Check for outliers in numeric columns using IQR method
  numeric_cols <- sapply(df, is.numeric)
  # remove the fraud_bool column from numeric_cols
  numeric_cols["fraud_bool"] <- FALSE
  outlier_info <- lapply(df[, numeric_cols], function(x) {
    x_no_na <- x[!is.na(x)]
    Q1 <- quantile(x_no_na, 0.25)
    Q3 <- quantile(x_no_na, 0.75)
    IQR <- Q3 - Q1
    lower_bound <- Q1 - 1.5 * IQR
    upper_bound <- Q3 + 1.5 * IQR
    outliers <- x_no_na[x_no_na < lower_bound | x_no_na > upper_bound]
    # return(list(lower_bound = lower_bound, upper_bound = upper_bound, outliers = outliers))
    return(list(lower_bound = lower_bound, upper_bound = upper_bound))
  })

  print("Outlier information for numeric columns:")
  print(outlier_info)
  return(df)
}


In [27]:
# Data Preprocessing based on this file https://github.com/feedzai/bank-account-fraud/blob/main/documents/datasheet.pdf

# Categorical Factors
categorical_factor_vars <- c(
  "payment_type",
  "employment_status",
  "housing_status",
  "source",
  "device_os"
)

# Y/N Factors
binary_factor_vars <- c(
  "email_is_free",
  "phone_home_valid",
  "phone_mobile_valid",
  "has_other_cards",
  "foreign_request",
  "keep_alive_session"
)

precessData <- function(df) {
  # Convert categorical variables to factors
  for (v in categorical_factor_vars) {
    df[[v]] <- as.factor(df[[v]])
  }

  # Convert Binary variables to factors
  for (v in binary_factor_vars) {
    df[[v]] <- factor(
      df[[v]],
      levels = c(0, 1),
      labels = c("No", "Yes")
    )
  }

  # Get da numeric variables
  numeric_vars <- names(df)[sapply(df, is.numeric)]

  # remove fraud_bool and month from numeric_vars
  numeric_predictors <- setdiff(numeric_vars, c("fraud_bool", "month"))
  scaling_parameters <- df %>%
    summarise(across(all_of(numeric_predictors),
      list(mean = mean, sd = sd),
      na.rm = TRUE
    ))

  # Normalize numeric variables
  baf_labeled_scaled <- df %>%
    mutate(across(
      all_of(numeric_predictors),
      ~ {
        mu <- mean(.x, na.rm = TRUE)
        sigma <- sd(.x, na.rm = TRUE)
        if (is.na(sigma) || sigma == 0) {
          ifelse(is.na(.x), NA_real_, 0)
        } else {
          (.x - mu) / sigma
        }
      }
    ))

  return(baf_labeled_scaled)
}


In [28]:
saveAsRDS <- function(df, filePath) {
  saveRDS(df, file = filePath)
  print(paste("Data saved as RDS file at:", filePath))
}


In [29]:
# Over Sample of the fraud instances
overSample_df <- function(df, train_p = 0.70) {
  if (!"fraud_bool" %in% names(df)) {
    stop("fraud_bool column not found in input dataframe.")
  }

  if (any(is.na(df$fraud_bool))) {
    stop("fraud_bool contains NA values. Clean target column before oversampling.")
  }

  if (!all(df$fraud_bool %in% c(0, 1))) {
    stop("fraud_bool must contain only 0/1 values.")
  }

  # Same prep logic as your oversampling script
  baf_model <- df %>%
    mutate(
      fraud_status = ifelse(fraud_bool == 1, "Fraud", "Not_Fraud"),
      fraud_status = factor(fraud_status, levels = c("Fraud", "Not_Fraud"))
    ) %>%
    select(-fraud_bool) %>%
    mutate(across(where(is.character), as.factor))

  # Same split logic
  train_index <- createDataPartition(
    baf_model$fraud_status,
    p = train_p,
    list = FALSE
  )

  train_data <- baf_model[train_index, , drop = FALSE]

  # Same oversampling logic
  train_data_over <- upSample(
    x = train_data %>% select(-fraud_status),
    y = train_data$fraud_status,
    yname = "fraud_status"
  )

  # Optional consistent level order used in your other script
  train_data_over$fraud_status <- factor(
    train_data_over$fraud_status,
    levels = c("Not_Fraud", "Fraud")
  )

  return(train_data_over)
}


In [30]:
# Prep for Regression Analysis

# CSV Files
dataDir <- "./Data/BankData"
dataFiles <- list.files(dataDir, pattern = "\\.csv$", full.names = TRUE)

out_dir <- "./Data/BankData/Processed"
if (!dir.exists(out_dir)) {
  dir.create(out_dir, recursive = TRUE)
}

for (file in dataFiles) {
  print(sprintf("--------------- FILE %s START ---------------", file))
  df_file <- readr::read_csv(file, show_col_types = FALSE)
  baselineInfo(df_file, file)
  checkFraudData(df_file)
  df_file <- dataQualityCheck(df_file)
  processed_df <- precessData(df_file)
  head(processed_df)
  saveAsRDS(processed_df, filePath = paste0(out_dir, "/", basename(file), "_PCD.rds"))
  overSampled_df <- overSample_df(processed_df)
  saveAsRDS(overSampled_df, filePath = paste0(out_dir, "/", basename(file), "_PCD_OS.rds"))
  print(sprintf("--------------- FILE %s DONE ---------------", file))
}


[1] "--------------- FILE ./Data/BankData/Base.csv START ---------------"
[1] "File: ./Data/BankData/Base.csv"
[1] "Number of rows: 1000000"
[1] "Number of columns: 32"
[1] "Column names: fraud_bool, income, name_email_similarity, prev_address_months_count, current_address_months_count, customer_age, days_since_request, intended_balcon_amount, payment_type, zip_count_4w, velocity_6h, velocity_24h, velocity_4w, bank_branch_count_8w, date_of_birth_distinct_emails_4w, employment_status, credit_risk_score, email_is_free, housing_status, phone_home_valid, phone_mobile_valid, bank_months_count, has_other_cards, proposed_credit_limit, foreign_request, source, session_length_in_minutes, device_os, keep_alive_session, device_distinct_emails_8w, device_fraud_count, month"
[1] "Data types: numeric, numeric, numeric, numeric, numeric, numeric, numeric, numeric, character, numeric, numeric, numeric, numeric, numeric, numeric, character, numeric, numeric, character, numeric, numeric, numeric, numeri

In [31]:
print("All files processed and saved as RDS.")


[1] "All files processed and saved as RDS."
